In [15]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split as tts
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
import pickle

new_df = pd.read_csv('Diabetes_clean.csv')

x = new_df.drop(columns=["Outcome"])
y = new_df.Outcome

num_col = x.select_dtypes(include=['number']).columns
cat_col = x.select_dtypes(include=['object', 'category']).columns

preprocessing = ColumnTransformer([
    ('num', StandardScaler(), num_col),
    ('cat', OrdinalEncoder(categories=[
        ['Low (Dangerous)', 'Normal (Healthy)', 'Pre-diabetes (Risk zone)', 'High (Diabetes)'],
        ['Low', 'Normal', 'High'],
        ['Low', 'Normal', 'High'],
        ['Underweight', 'Normal', 'Overweight', 'Obese'],
        ['Low', 'Medium', 'High'],
        ['Young', 'Adult', 'Middle_Age', 'Senior']
    ]), cat_col)
])

x_train,x_test,y_train,y_test=tts(x,y,test_size=0.25, random_state=42)

rf_pipe = Pipeline([
    ("preprocessing", preprocessing),
    ("model", RandomForestClassifier(random_state=42))
])

param = {
    'model__criterion': ['entropy', 'gini', 'log_loss'],
    'model__max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'model__min_samples_split': [2, 3, 4] 
}

GridCV1 = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param,
    cv=5,
    n_jobs=-1,
    verbose=1
)

GridCV1.fit(x_train, y_train)

# pickle.dump(GridCV1, open("model.pkl", "wb"))

Fitting 5 folds for each of 90 candidates, totalling 450 fits
